<a href="https://colab.research.google.com/github/himanshubhimte69/AgriLite-A-Lightweight-Multi-Crop-Disease-Detector-Across-Diverse-Conditions/blob/main/Plantdoc_EfficientNetB4_FineTuneipynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!git clone https://github.com/pratikkayal/PlantDoc-Dataset.git
!mv PlantDoc-Dataset /content/PlantDoc
!ls /content/PlantDoc

Cloning into 'PlantDoc-Dataset'...
remote: Enumerating objects: 2670, done.
remote: Counting objects: 100% (35/35), done.
remote: Compressing objects: 100% (13/13), done.
remote: Total 2670 (delta 22), reused 22 (delta 22), pack-reused 2635 (from 1)
Receiving objects: 100% (2670/2670), 932.92 MiB | 49.31 MiB/s, done.
Resolving deltas: 100% (24/24), done.
Updating files: 100% (2581/2581), done.
LICENSE.txt  PlantDoc_Examples.png  README.md  test  train


In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models, regularizers
from tensorflow.keras.applications import EfficientNetB4, efficientnet
from tensorflow.keras.preprocessing import image_dataset_from_directory
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, cohen_kappa_score
from sklearn.preprocessing import label_binarize
import random

In [ ]:
img_size = (380, 380)
batch_size = 16

train_ds_full = image_dataset_from_directory(
    "/content/PlantDoc/train",
    image_size=img_size,
    batch_size=batch_size
)

test_ds = image_dataset_from_directory(
    "/content/PlantDoc/test",
    image_size=img_size,
    batch_size=batch_size
)

class_names = train_ds_full.class_names
num_classes = len(class_names)

# Split train into train + val (20%)
val_batches = tf.data.experimental.cardinality(train_ds_full) // 5
val_ds = train_ds_full.take(val_batches)
train_ds = train_ds_full.skip(val_batches)

print("Classes:", class_names)
print("Num Classes:", num_classes)

Found 2342 files belonging to 28 classes.
Found 236 files belonging to 27 classes.
Classes: ['Apple Scab Leaf', 'Apple leaf', 'Apple rust leaf', 'Bell_pepper leaf', 'Bell_pepper leaf spot', 'Blueberry leaf', 'Cherry leaf', 'Corn Gray leaf spot', 'Corn leaf blight', 'Corn rust leaf', 'Peach leaf', 'Potato leaf early blight', 'Potato leaf late blight', 'Raspberry leaf', 'Soyabean leaf', 'Squash Powdery mildew leaf', 'Strawberry leaf', 'Tomato Early blight leaf', 'Tomato Septoria leaf spot', 'Tomato leaf', 'Tomato leaf bacterial spot', 'Tomato leaf late blight', 'Tomato leaf mosaic virus', 'Tomato leaf yellow virus', 'Tomato mold leaf', 'Tomato two spotted spider mites leaf', 'grape leaf', 'grape leaf black rot']
Num Classes: 28


In [ ]:
# -----------------------------
# Preprocessing + Augmentation
# -----------------------------
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
    layers.RandomContrast(0.1),
])

preprocess_input = efficientnet.preprocess_input

train_ds = train_ds.map(lambda x, y: (preprocess_input(data_augmentation(x)), y))
val_ds   = val_ds.map(lambda x, y: (preprocess_input(x), y))
test_ds  = test_ds.map(lambda x, y: (preprocess_input(x), y))

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(AUTOTUNE)
val_ds   = val_ds.prefetch(AUTOTUNE)
test_ds  = test_ds.prefetch(AUTOTUNE)

In [ ]:
# -----------------------------
# Base Model (Frozen Training)
# -----------------------------
base_model = EfficientNetB4(
    include_top=False,
    weights="imagenet",
    input_shape=(380, 380, 3)
)
base_model.trainable = False

model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.5),
    layers.Dense(256, activation="relu", kernel_regularizer=regularizers.l2(0.001)),
    layers.Dropout(0.4),
    layers.Dense(num_classes, activation="softmax")
])

def one_hot_map(images, labels):
    return images, tf.one_hot(labels, depth=num_classes)

train_ds_onehot = train_ds.map(one_hot_map)
val_ds_onehot   = val_ds.map(one_hot_map)
test_ds_onehot  = test_ds.map(one_hot_map)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
    metrics=["accuracy"]
)

history = model.fit(
    train_ds_onehot,
    validation_data=val_ds_onehot,
    epochs=12
)

71686520/71686520 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Epoch 1/12
118/118 ━━━━━━━━━━━━━━━━━━━━ 194s 1s/step - accuracy: 0.1914 - loss: 3.3987 - val_accuracy: 0.4375 - val_loss: 2.6268
Epoch 2/12
118/118 ━━━━━━━━━━━━━━━━━━━━ 117s 718ms/step - accuracy: 0.4320 - loss: 2.6109 - val_accuracy: 0.5086 - val_loss: 2.3739
Epoch 3/12
118/118 ━━━━━━━━━━━━━━━━━━━━ 141s 704ms/step - accuracy: 0.4842 - loss: 2.4578 - val_accuracy: 0.5043 - val_loss: 2.3223
Epoch 4/12
118/118 ━━━━━━━━━━━━━━━━━━━━ 142s 706ms/step - accuracy: 0.5244 - loss: 2.3353 - val_accuracy: 0.5517 - val_loss: 2.2640
Epoch 5/12
118/118 ━━━━━━━━━━━━━━━━━━━━ 142s 713ms/step - accuracy: 0.5178 - loss: 2.2890 - val_accuracy: 0.5388 - val_loss: 2.2222
Epoch 6/12
118/118 ━━━━━━━━━━━━━━━━━━━━ 142s 716ms/step - accuracy: 0.5796 - loss: 2.2186 - val_accuracy: 0.5690 - val_loss: 2.1837
Epoch 7/12
118/118 ━━━━━━━━━━━━━━━━━━━━ 90s 708ms/step - accuracy: 0.5739 - loss: 2.1733 - val_accuracy: 0.5647 - val_loss: 2.1907
Epoch 8/12
118/118 ━━━━━━━━━━

In [ ]:
# -----------------------------
# Randomized Parameter Optimization for Fine-tuning
# -----------------------------
param_dist = {
    "learning_rate": [1e-4, 5e-5, 1e-5],
    "dropout_dense": [0.3, 0.4, 0.5],
    "dropout_final": [0.3, 0.4, 0.5],
    "dense_units": [128, 256, 512],
    "weight_decay": [1e-3, 1e-4, 1e-5],
}

n_iter = 3  # number of random trials
best_val_acc = 0
best_params = None
best_model = None

for i in range(n_iter):
    print(f"\n Fine-tuning Trial {i+1}/{n_iter}")
    params = {k: random.choice(v) for k, v in param_dist.items()}
    print("Params:", params)

    base_model = EfficientNetB4(
        include_top=False,
        weights="imagenet",
        input_shape=(380, 380, 3)
    )
    base_model.trainable = True
    for layer in base_model.layers[:300]:
        layer.trainable = False

    candidate = models.Sequential([
        base_model,
        layers.GlobalAveragePooling2D(),
        layers.Dropout(params["dropout_dense"]),
        layers.Dense(params["dense_units"], activation="relu",
                     kernel_regularizer=regularizers.l2(params["weight_decay"])),
        layers.Dropout(params["dropout_final"]),
        layers.Dense(num_classes, activation="softmax")
    ])

    candidate.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=params["learning_rate"]),
        loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
        metrics=["accuracy"]
    )

    history_candidate = candidate.fit(
        train_ds_onehot,
        validation_data=val_ds_onehot,
        epochs=8,  # short runs for comparison
        verbose=1
    )

    val_acc = max(history_candidate.history["val_accuracy"])
    print(f"Validation Accuracy: {val_acc:.4f}")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_params = params
        best_model = candidate

print("\n Best Params for Fine-tuning:", best_params)


 Fine-tuning Trial 1/3
Params: {'learning_rate': 5e-05, 'dropout_dense': 0.5, 'dropout_final': 0.5, 'dense_units': 512, 'weight_decay': 0.001}
Epoch 1/8
118/118 ━━━━━━━━━━━━━━━━━━━━ 225s 1s/step - accuracy: 0.0569 - loss: 4.1682 - val_accuracy: 0.2198 - val_loss: 3.8801
Epoch 2/8
118/118 ━━━━━━━━━━━━━━━━━━━━ 106s 739ms/step - accuracy: 0.1954 - loss: 3.8149 - val_accuracy: 0.3836 - val_loss: 3.4257
Epoch 3/8
118/118 ━━━━━━━━━━━━━━━━━━━━ 142s 733ms/step - accuracy: 0.3056 - loss: 3.4310 - val_accuracy: 0.4957 - val_loss: 2.9966
Epoch 4/8
118/118 ━━━━━━━━━━━━━━━━━━━━ 143s 744ms/step - accuracy: 0.4396 - loss: 3.0416 - val_accuracy: 0.5431 - val_loss: 2.7326
Epoch 5/8
118/118 ━━━━━━━━━━━━━━━━━━━━ 96s 771ms/step - accuracy: 0.5075 - loss: 2.8146 - val_accuracy: 0.5819 - val_loss: 2.5583
Epoch 6/8
118/118 ━━━━━━━━━━━━━━━━━━━━ 93s 750ms/step - accuracy: 0.5650 - loss: 2.6065 - val_accuracy: 0.6530 - val_loss: 2.4083
Epoch 7/8
118/118 ━━━━━━━━━━━━━━━━━━━━ 94s 743ms/step - accuracy: 0.6272 - 

In [ ]:
# -----------------------------
# Final Fine-tuning with Best Model
# -----------------------------
history_finetune = best_model.fit(
    train_ds_onehot,
    validation_data=val_ds_onehot,
    epochs=30
)

Epoch 1/30
118/118 ━━━━━━━━━━━━━━━━━━━━ 94s 748ms/step - accuracy: 0.8917 - loss: 1.6894 - val_accuracy: 0.7802 - val_loss: 1.9927
Epoch 2/30
118/118 ━━━━━━━━━━━━━━━━━━━━ 146s 795ms/step - accuracy: 0.9229 - loss: 1.6291 - val_accuracy: 0.7672 - val_loss: 2.0111
Epoch 3/30
118/118 ━━━━━━━━━━━━━━━━━━━━ 94s 758ms/step - accuracy: 0.9324 - loss: 1.5873 - val_accuracy: 0.7823 - val_loss: 1.9735
Epoch 4/30
118/118 ━━━━━━━━━━━━━━━━━━━━ 141s 749ms/step - accuracy: 0.9472 - loss: 1.5465 - val_accuracy: 0.8082 - val_loss: 1.8947
Epoch 5/30
118/118 ━━━━━━━━━━━━━━━━━━━━ 142s 747ms/step - accuracy: 0.9610 - loss: 1.4996 - val_accuracy: 0.8017 - val_loss: 1.9019
Epoch 6/30
118/118 ━━━━━━━━━━━━━━━━━━━━ 141s 737ms/step - accuracy: 0.9546 - loss: 1.4857 - val_accuracy: 0.8082 - val_loss: 1.8752
Epoch 7/30
118/118 ━━━━━━━━━━━━━━━━━━━━ 143s 744ms/step - accuracy: 0.9561 - loss: 1.4640 - val_accuracy: 0.8168 - val_loss: 1.8180
Epoch 8/30
118/118 ━━━━━━━━━━━━━━━━━━━━ 142s 742ms/step - accuracy: 0.9632 - l

In [ ]:
# -----------------------------
# Temperature Scaling & Evaluation
# -----------------------------
TEMPERATURE = 2.0

def predict_with_temperature(model, dataset, T=1.0):
    all_probs, all_labels = [], []
    for images, labels in dataset:
        images = tf.cast(images, tf.float32)
        logits = model(images, training=False)
        scaled_logits = logits / T
        probs = tf.nn.softmax(scaled_logits).numpy()
        all_probs.extend(probs)
        all_labels.extend(labels.numpy())
    return np.array(all_labels), np.array(all_probs)

y_true, y_probs = predict_with_temperature(best_model, test_ds_onehot, T=TEMPERATURE)
y_pred = np.argmax(y_probs, axis=1)
y_true_int = np.argmax(y_true, axis=1)

present_classes = np.unique(y_true_int)
if len(present_classes) > 1:
    y_true_bin_present = label_binarize(y_true_int, classes=present_classes)
    y_probs_present = y_probs[:, present_classes]
    auc = roc_auc_score(
        y_true_bin_present,
        y_probs_present,
        average="macro",
        multi_class="ovr"
    )
else:
    print("⚠️ Only one class present in test set, AUC not defined.")
    auc = np.nan

accuracy = accuracy_score(y_true_int, y_pred)
precision = precision_score(y_true_int, y_pred, average="weighted", zero_division=0)
recall = recall_score(y_true_int, y_pred, average="weighted", zero_division=0)
f1 = f1_score(y_true_int, y_pred, average="weighted", zero_division=0)
kappa = cohen_kappa_score(y_true_int, y_pred)

print("\n Evaluation Metrics After Calibration:")
print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1 Score : {f1:.4f}")
print(f"AUC      : {auc:.4f}")
print(f"Kappa    : {kappa:.4f}")


 Evaluation Metrics After Calibration:
Accuracy : 0.6059
Precision: 0.6507
Recall   : 0.6059
F1 Score : 0.6040
AUC      : 0.8962
Kappa    : 0.5914


In [ ]:
import tensorflow as tf
import numpy as np
import os

In [ ]:
model.save("best_model.h5")